# Exercise MegaDetector on Local Images

This notebook runs MegaDetector on a local image folder. The folder is
resolved automatically from the machine detected in Step 1 (the `data_root`
for this box, e.g. `/fs/work/Pictures` on the Spark Station).

It is safe to run even when the folder is currently empty.

## Step 1 — Define paths

In [ ]:
# find the machine we're running on, and set the repo/data/results roots accordingly. 
# This is used by the %%bash cells in the notebooks to set up the environment.

from pathlib import Path
import subprocess
import socket
import os

# Set MACHINE explicitly to force a specific MACHINE_PATHS row, or leave as
# None (optionally via the MACHINE env var) to auto-detect this box.
MACHINE = os.environ.get("MACHINE") or None

MACHINE_PATHS = {
    "spark": {
        "repo_root":    "/fs/work/git/ebio/project-id",
        "data_root":    "/fs/work/Pictures",
        "results_root": "/fs/work/Results",
    },
    "bigmac": {
        "repo_root":    "/Users/elhorte/git/Amryth/omicsaf",
        "data_root":    "/Users/Shared/Amryth/Data/omicsaf",
        "results_root": "/Users/Shared/Amryth/Results/omicsaf/elhorte",
    },
    "bigmacx": {
        "repo_root":    "/Users/elhorte/git/Amryth/omicsaf",
        "data_root":    "/Volumes/BigMacX/Amryth/Data/omicsaf",
        "results_root": "/Volumes/BigMacX/Amryth/Results/omicsaf",
    },
    "MacBook": {
        "repo_root":    "/Users/elhorte/git/Amryth/omicsaf",
        "data_root":    "/Users/elhorte/Data/Amryth/omicsaf",  
        "results_root": "/Users/elhorte/Data/Amryth/omicsaf", 
    },
    "i9-14": {
        "repo_root":    "",   # TODO: set repo root for i9-14
        "data_root":    "",   # TODO: set data root for i9-14
        "results_root": "",   # TODO: set results root for i9-14
    },
}

def _detect_machine(table):
    """Pick the MACHINE_PATHS row for the box we're running on (see top note)."""
    override = os.environ.get("MACHINE_OVERRIDE")
    if override:
        if override not in table:
            raise ValueError(
                f"MACHINE_OVERRIDE={override!r} is not in MACHINE_PATHS "
                f"{sorted(table)}")
        return override, "MACHINE_OVERRIDE"
    # Filesystem truth: a row is viable only if BOTH its roots exist on this box.
    viable = [
        m for m, p in table.items()
        if p.get("repo_root") and os.path.isdir(p["repo_root"])
        and p.get("data_root") and os.path.isdir(p["data_root"])
    ]
    if len(viable) == 1:
        return viable[0], "filesystem"
    host = socket.gethostname().lower()
    # Hostname disambiguates when 0 or >1 rows' paths are visible.
    for m in (viable or list(table)):
        if m.lower() in host:
            return m, f"hostname({host})"
    if len(viable) > 1:
        return viable[0], f"filesystem(ambiguous:{viable})"
    raise RuntimeError(
        "Could not auto-detect MACHINE: no MACHINE_PATHS row has both its "
        f"repo_root and data_root present on host {host!r}, and no row name "
        "matches the hostname. Fill in MACHINE_PATHS or "
        f"`export MACHINE_OVERRIDE=<name>` (known: {sorted(table)})."
    )

if MACHINE is None:
    MACHINE, _machine_src = _detect_machine(MACHINE_PATHS)
else:
    if MACHINE not in MACHINE_PATHS:
        raise ValueError(
            f"MACHINE={MACHINE!r} is not a key of MACHINE_PATHS "
            f"(known: {sorted(MACHINE_PATHS)})"
        )
    _machine_src = "manual"

os.environ["MACHINE"] = MACHINE            # OUTPUT for the %%bash cells
_paths = MACHINE_PATHS[MACHINE]
_missing = [k for k, v in _paths.items() if not v]
if _missing:
    raise ValueError(
        f"MACHINE={MACHINE!r} is missing path(s) {_missing} in MACHINE_PATHS; "
        "fill them in before running.")

REPO         = _paths["repo_root"]
R            = _paths["data_root"]
RESULTS_ROOT = _paths["results_root"]

repo_root = Path(REPO)
md_root = repo_root / "third-party" / "eb_megadetector"
image_dir = Path(R)                          # data_root for this machine
output_dir = Path(RESULTS_ROOT) / "megadetector"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Machine: {MACHINE} (via {_machine_src})")
print(f"Repository root: {repo_root}")
print(f"MegaDetector path: {md_root}")
print(f"Image directory: {image_dir}")
print(f"Output directory: {output_dir}")

## Step 2 — Check image folder contents

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted([p for p in image_dir.glob("**/*") if p.is_file() and p.suffix.lower() in image_exts])

print(f"Found {len(images)} image(s)")
for p in images[:20]:
    print(" -", p)

if not images:
    print("\nNo images found yet. Populate the folder and rerun this notebook.")

## Step 3 — Build an image list file for MegaDetector batch inference

In [ ]:
image_list_file = output_dir / "image_list.txt"
image_list_file.write_text("\n".join(str(p) for p in images), encoding="utf-8")
print(f"Wrote: {image_list_file}")

## Step 4 — Run MegaDetector (batch mode)

This uses the MegaDetector batch detection script and writes a JSON results file.

> If your local MegaDetector checkout uses a different entry script, update the command in the next cell accordingly.

In [ ]:
import subprocess
import sys

results_json = output_dir / "megadetector_results.json"
batch_script = md_root / "megadetector" / "detection" / "run_detector_batch.py"

if not md_root.exists():
    raise FileNotFoundError(f"MegaDetector repo not found: {md_root}")
if not batch_script.exists():
    raise FileNotFoundError(f"Batch script not found: {batch_script}")
if len(images) == 0:
    raise RuntimeError("No images available for inference. Add images and rerun.")

cmd = [
    sys.executable,
    str(batch_script),
    "MDV5A",
    str(image_list_file),
    str(results_json),
    "--recursive",
]

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(f"\nDone. Results saved to: {results_json}")

## Step 5 — Preview detections summary

In [ ]:
import json
from collections import Counter

results_json = output_dir / "megadetector_results.json"
data = json.loads(results_json.read_text(encoding="utf-8"))
images_data = data.get("images", [])

print(f"Images in results: {len(images_data)}")

detections_per_image = Counter()
for item in images_data:
    detections_per_image[len(item.get("detections", []))] += 1

print("Detection count distribution (detections -> number of images):")
for k in sorted(detections_per_image):
    print(f"  {k} -> {detections_per_image[k]}")

## Step 6 — Visualize detections (bounding-box QA)

Draws MegaDetector bounding boxes on detected images and saves annotated copies to
an `annotated/` subfolder of the Step 1 `output_dir` (e.g.
`/fs/work/Results/megadetector/annotated` on the Spark Station).

Set `annotate_all = True` to annotate **every** successfully detected image; leave it
`False` to only process the first `max_preview` images. Regardless of the mode, up to
`max_preview` annotated images are shown inline so the notebook stays responsive.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display
import megadetector.visualization.visualization_utils as vis_utils

# Reuse output_dir from Step 1; results file from Step 4.
results_json = output_dir / "megadetector_results.json"
results = json.loads(results_json.read_text(encoding="utf-8"))
category_map = results.get("detection_categories", {})

# --- Options ---------------------------------------------------------------
confidence_threshold = 0.2   # only draw boxes at/above this confidence
annotate_all = True          # True: annotate every detected image; False: only a preview
max_preview = 6              # how many annotated images to show inline (either mode)
# ---------------------------------------------------------------------------

# Saves to the 'annotated' subfolder of output_dir (defined in Step 1).
annotated_dir = output_dir / "annotated"
annotated_dir.mkdir(parents=True, exist_ok=True)

# Pick a label font that exists on this machine (Linux/Spark, macOS, or the
# MegaDetector default). None lets MegaDetector fall back to its bundled font.
_font_candidates = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",          # Linux / Spark
    "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    "/System/Library/Fonts/Supplemental/Arial.ttf",            # macOS
]
label_font = next((f for f in _font_candidates if Path(f).exists()), None)

images_with_detections = [
    im for im in results.get("images", [])
    if not im.get("failure")
    and any(d.get("conf", 0) >= confidence_threshold for d in im.get("detections", []))
]

total = len(images_with_detections)
print(f"{total} image(s) have detections at conf >= {confidence_threshold}")
if not images_with_detections:
    print("Nothing to visualize yet. Run Step 4 on a folder that contains animals/people/vehicles.")

to_process = images_with_detections if annotate_all else images_with_detections[:max_preview]
print(f"Annotating {len(to_process)} image(s) "
      f"({'all detected' if annotate_all else 'preview only'}); "
      f"showing up to {max_preview} inline.")

saved = 0
for idx, im in enumerate(to_process):
    src = Path(im["file"])
    if not src.exists():
        print(f"  (skipped, missing file) {src}")
        continue

    image = vis_utils.load_image(str(src))
    vis_utils.render_detection_bounding_boxes(
        im["detections"], image,
        label_map=category_map,
        confidence_threshold=confidence_threshold,
        thickness=4,
        label_font=label_font,
        label_font_size=16,
    )

    out_path = annotated_dir / f"{src.stem}_annotated.png"
    image.save(out_path)
    saved += 1

    n_boxes = sum(1 for d in im["detections"] if d.get("conf", 0) >= confidence_threshold)
    print(f"  [{idx + 1}/{len(to_process)}] {src.name}: {n_boxes} box(es) -> {out_path.name}")

    if idx < max_preview:
        display(IPyImage(filename=str(out_path), width=700))

print(f"\nSaved {saved} annotated image(s) to: {annotated_dir}")